In [19]:
import tensorflow as ts
import numpy as np
import glob as glob
from PIL import Image
from tensorflow.keras import models
from tensorflow.keras import layers
from tensorflow.keras.layers import UpSampling2D

In [20]:
train_black = np.array([np.array(Image.open(p).convert('L').resize((96,96))) for p in glob.glob('data/train_black/*.jpg')])
train_color = np.array([np.array(Image.open(p).convert('RGB').resize((96,96))) for p in glob.glob('data/train_color/*.jpg')])

In [21]:
test_black = np.array([np.array(Image.open(p).convert('L').resize((96,96))) for p in glob.glob('data/test_black/*.jpg')])
test_color = np.array([np.array(Image.open(p).convert('RGB').resize((96,96))) for p in glob.glob('data/test_color/*.jpg')])

In [22]:
train_black = train_black.astype('float32') / 255
train_color = train_color.astype('float32') / 255
test_black = test_black.astype('float32') / 255
test_color = test_color.astype('float32') / 255

In [23]:
def main_model():
    
    model = models.Sequential()
    
    model.add(layers.Conv2D(32,(3,3), activation = 'relu', input_shape = (96,96,1), padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(2,2)))
    
    model.add(layers.Conv2D(64,(3,3), activation = 'relu', padding='same'))
    model.add(layers.MaxPooling2D(pool_size=(2,2)))
    
    model.add(layers.Conv2D(128,(3,3), activation = 'relu', padding='same'))
    model.add(layers.Conv2D(128,(3,3), activation = 'relu', padding='same'))

    model.add(UpSampling2D(size=(2,2)))
    
    model.add(layers.Conv2D(64,(3,3), activation = 'relu', padding='same'))
    
    model.add(layers.UpSampling2D(size=(2,2)))
    model.add(layers.Conv2D(32,(3,3), activation = 'relu', padding='same'))
    
    model.add(layers.Conv2D(3,(3,3), activation = 'sigmoid', padding='same'))
    
    return model

In [24]:
model = main_model()

model.compile(loss = 'mean_squared_error', optimizer = 'adam', metrics = ['mean_squared_error'])

In [25]:
X_train = train_black
y_train = train_color
X_test = test_black
y_test = test_color

history = model.fit(X_train, y_train, validation_data = (X_test, y_test) , epochs = 20)

Epoch 1/20
157/157 [==============================] - 59s 374ms/step - loss: 0.0201 - mean_squared_error: 0.0201 - val_loss: 0.0128 - val_mean_squared_error: 0.0128
Epoch 2/20
157/157 [==============================] - 63s 399ms/step - loss: 0.0117 - mean_squared_error: 0.0117 - val_loss: 0.0108 - val_mean_squared_error: 0.0108
Epoch 3/20
157/157 [==============================] - 109s 693ms/step - loss: 0.0108 - mean_squared_error: 0.0108 - val_loss: 0.0112 - val_mean_squared_error: 0.0112
Epoch 4/20
157/157 [==============================] - 64s 407ms/step - loss: 0.0103 - mean_squared_error: 0.0103 - val_loss: 0.0097 - val_mean_squared_error: 0.0097
Epoch 5/20
157/157 [==============================] - 67s 429ms/step - loss: 0.0100 - mean_squared_error: 0.0100 - val_loss: 0.0097 - val_mean_squared_error: 0.0097
Epoch 6/20
157/157 [==============================] - 67s 429ms/step - loss: 0.0097 - mean_squared_error: 0.0097 - val_loss: 0.0100 - val_mean_squared_error: 0.0100
Epoch 7/2

In [31]:
def colorize(path, model, size = (96,96)):
    
    image = Image.open(path).convert('L')
    image_original_size = image.size
    image = image.resize(size)

    arr = np.array(image).astype('float32') / 255
    arr = arr.reshape(1, size[0], size[1], 1)

    pred = model.predict(arr)
    pred_image = (pred[0]*255).astype('uint8')

    pred_image = Image.fromarray(pred_image)
    pred_image = pred_image.resize(image_original_size, Image.Resampling.BICUBIC)

    return pred_image

In [33]:
input_image = input('Enter the path to your black and white image: ')
output = colorize(input_image, model)
output.show()

Enter the path to your black and white image:  testImage.jpg


1/1 [==============================] - 0s 19ms/step
